### This notebook demonstrates a typical ETL pipeline, including reading data from the bronze layer, transforming it, and loading the clean data into the silver layer.

### Read the data from the bronze layer

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql.functions import *

In [0]:
df = spark.table("datalakehouse.bronze.bronze_stock_prices")
display(df)


### Data Transformation

Let's perform the possible transformations:
- Rename the columns for clarity
- Standardize columns format and values in each column
- Trim white spaces in columns 
- Remove duplicate records


In [0]:
df.printSchema()

In [0]:
# let's see if there are any null values
from pyspark.sql.functions import col

display(
  df.select(
    [col(c).isNull().alias(f"{c}_isnull") for c in df.columns]
  )
)

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name.lower(), trim(col(field.name.lower())))
display(df)

### Load the clean data into the silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("datalakehouse.sliver.silver_stock_prices")

In [0]:
%sql 
select * from datalakehouse.sliver.silver_stock_prices